# 800 — External Transcriptomic Program Replication in CPTAC-3

## Scientific Objective

Evaluate the external reproducibility of three frozen Phase 4 consensus transcriptomic programs in primary tumors from CPTAC-3.

The primary scientific objective is to assess whether TCGA reference gene–gene coexpression patterns are preserved in CPTAC-3 within matched cancer lineages. Program-score projection and internal coexpression coherence provide complementary characterization but do not independently establish external replication.

## Data Sources

- **External cohort:** 727 proposed CPTAC-3 primary-tumor RNA-seq cases across seven harmonized cancer lineages, defined by Notebook 113.
- **Reference programs:** `CONSENSUS_TX_01`, `CONSENSUS_TX_02`, and `CONSENSUS_TX_03`, constructed and frozen in Phase 4.
- **Feature universe:** 2,389 shared genes with immutable identities, ordering, signs, and consensus weights.
- **Expression representation:** GDC STAR Counts (`unstranded`), independently normalized using TMM and log2-CPM.

## Analytical Scope

The workflow verifies upstream inputs, prepares the external expression matrix, projects frozen transcriptomic programs, and evaluates lineage-specific structural reproducibility under prespecified statistical procedures.

Cohort eligibility, transcriptomic feature selection, program construction, and loading orientations are not re-estimated.

## Scientific Boundaries

CPTAC-3 externality is supported by cohort provenance; individual donor non-overlap with TCGA has not been independently established. Read-level sequencing QC and cross-platform comparability remain explicit limitations.

Weighted edge-pattern concordance (T) is the primary descriptive statistic for the lineage-matched comparison between TCGA and CPTAC-3.

Computable program scores and descriptive coexpression patterns are not sufficient evidence of external replication. Any supported findings represent computational associations, not causal mechanisms, clinical resistance prediction, or therapeutic validation.

In [1]:
# =============================================================================
# Imports
# =============================================================================

import json
import h5py

import numpy as np
import pandas as pd
from IPython.display import display

from pancancer_epigenetics.utils.artifact_registry import (
    load_artifact_registry,
    resolve_artifact_path,
)
from pancancer_epigenetics.utils.file_checks import calculate_sha256
from pancancer_epigenetics.utils.paths import Paths, project_relative_path

In [2]:
# =============================================================================
# Resolve external cohort and frozen reference inputs
# =============================================================================

SOURCE_INPUTS = {
    "manifest": Paths.metadata / "113_source_provenance_manifest.json",
    "proposed_cohort": Paths.metadata / "113_proposed_external_source_cohort.csv",
    "candidate_eligibility": Paths.metadata / "113_primary_candidate_case_eligibility.csv",
    "raw_count_compatibility": Paths.metadata / "113_raw_count_compatibility.csv",
}

FROZEN_ARTIFACT_IDS = (
    "phase4.400.cross_system_shared_gene_universe",
    "phase4.401.consensus_transcriptomic_gene_weights",
    "phase4.401.consensus_transcriptomic_program_catalog",
    "phase4.401.consensus_construction_metadata",
)

artifact_registry = load_artifact_registry()

reference_paths = {
    artifact_id: resolve_artifact_path(artifact_registry, artifact_id)
    for artifact_id in FROZEN_ARTIFACT_IDS
}

print("Notebook 113 source inputs:")
for name, path in SOURCE_INPUTS.items():
    print(f"  {name}: {project_relative_path(path)}")

print("\nFrozen Phase 4 reference artifacts:")
for artifact_id, path in reference_paths.items():
    print(f"  {artifact_id}: {project_relative_path(path)}")

Notebook 113 source inputs:
  manifest: data/interim/metadata/113_source_provenance_manifest.json
  proposed_cohort: data/interim/metadata/113_proposed_external_source_cohort.csv
  candidate_eligibility: data/interim/metadata/113_primary_candidate_case_eligibility.csv
  raw_count_compatibility: data/interim/metadata/113_raw_count_compatibility.csv

Frozen Phase 4 reference artifacts:
  phase4.400.cross_system_shared_gene_universe: data/processed/consensus_programs/400_cross_system_shared_gene_universe.csv
  phase4.401.consensus_transcriptomic_gene_weights: data/processed/consensus_programs/401_consensus_transcriptomic_gene_weights.csv
  phase4.401.consensus_transcriptomic_program_catalog: data/processed/consensus_programs/401_consensus_transcriptomic_program_catalog.csv
  phase4.401.consensus_construction_metadata: data/processed/consensus_programs/401_consensus_construction_metadata.json


In [3]:
# =============================================================================
# Load and verify external cohort provenance
# =============================================================================

with SOURCE_INPUTS["manifest"].open(encoding="utf-8") as handle:
    provenance_113 = json.load(handle)

if provenance_113["schema_version"] != 1:
    raise ValueError("Unsupported Notebook 113 manifest schema")

for name in (
    "proposed_cohort",
    "candidate_eligibility",
    "raw_count_compatibility",
):
    path = SOURCE_INPUTS[name]
    recorded = provenance_113["derived_outputs"][name]

    if recorded["path"] != project_relative_path(path):
        raise ValueError(f"Unexpected source artifact path: {name}")

    if calculate_sha256(path) != recorded["sha256"]:
        raise ValueError(f"Source artifact SHA-256 mismatch: {name}")

    print(f"{name}: SHA-256 VERIFIED")

print("\nNotebook 113 provenance verified.")

proposed_cohort: SHA-256 VERIFIED
candidate_eligibility: SHA-256 VERIFIED
raw_count_compatibility: SHA-256 VERIFIED

Notebook 113 provenance verified.


In [4]:
# =============================================================================
# Load external cohort and eligibility tables
# =============================================================================

cohort_113 = pd.read_csv(SOURCE_INPUTS["proposed_cohort"])
eligibility_113 = pd.read_csv(SOURCE_INPUTS["candidate_eligibility"])
compatibility_113 = pd.read_csv(SOURCE_INPUTS["raw_count_compatibility"])

source_tables = {
    "proposed_cohort": cohort_113,
    "candidate_eligibility": eligibility_113,
    "raw_count_compatibility": compatibility_113,
}

for name, frame in source_tables.items():
    recorded = provenance_113["derived_outputs"][name]
    observed_shape = frame.shape
    expected_shape = (recorded["rows"], recorded["columns"])

    if observed_shape != expected_shape:
        raise ValueError(f"Unexpected table dimensions: {name}")

    print(f"{name}: {observed_shape[0]:,} rows × {observed_shape[1]} columns")

proposed_cohort: 727 rows × 22 columns
candidate_eligibility: 1,082 rows × 30 columns
raw_count_compatibility: 727 rows × 10 columns


In [5]:
# =============================================================================
# Load frozen Phase 4 reference artifacts
# =============================================================================

for artifact_id, path in reference_paths.items():
    expected_sha256 = artifact_registry["artifacts"][artifact_id]["sha256"]

    if calculate_sha256(path) != expected_sha256:
        raise ValueError(f"Artifact SHA-256 mismatch: {artifact_id}")

    print(f"{artifact_id}: SHA-256 VERIFIED")

shared_gene_universe = pd.read_csv(
    reference_paths["phase4.400.cross_system_shared_gene_universe"]
)

consensus_gene_weights = pd.read_csv(
    reference_paths["phase4.401.consensus_transcriptomic_gene_weights"]
)

consensus_catalog = pd.read_csv(
    reference_paths["phase4.401.consensus_transcriptomic_program_catalog"]
)

with reference_paths[
    "phase4.401.consensus_construction_metadata"
].open(encoding="utf-8") as handle:
    consensus_metadata = json.load(handle)

print("\nFrozen Phase 4 references loaded.")
print(f"Shared genes: {len(shared_gene_universe):,}")
print(f"Consensus weight records: {len(consensus_gene_weights):,}")
print(f"Consensus programs: {len(consensus_catalog)}")

phase4.400.cross_system_shared_gene_universe: SHA-256 VERIFIED
phase4.401.consensus_transcriptomic_gene_weights: SHA-256 VERIFIED
phase4.401.consensus_transcriptomic_program_catalog: SHA-256 VERIFIED
phase4.401.consensus_construction_metadata: SHA-256 VERIFIED

Frozen Phase 4 references loaded.
Shared genes: 2,389
Consensus weight records: 7,167
Consensus programs: 3


In [6]:
# =============================================================================
# Align frozen consensus gene weights
# =============================================================================

PROGRAM_IDS = (
    "CONSENSUS_TX_01",
    "CONSENSUS_TX_02",
    "CONSENSUS_TX_03",
)

gene_order = shared_gene_universe["gene_symbol"]

if set(consensus_gene_weights["gene_symbol"]) != set(gene_order):
    raise ValueError("Consensus weights do not match the frozen gene universe")

if set(consensus_gene_weights["consensus_program_id"]) != set(PROGRAM_IDS):
    raise ValueError("Unexpected consensus program identities")

consensus_weight_matrix = (
    consensus_gene_weights.pivot(
        index="gene_symbol",
        columns="consensus_program_id",
        values="consensus_weight",
    )
    .reindex(index=gene_order, columns=PROGRAM_IDS)
)

if consensus_weight_matrix.isna().any().any():
    raise ValueError("Missing consensus weights after gene alignment")

if not np.isfinite(consensus_weight_matrix.to_numpy()).all():
    raise ValueError("Nonfinite consensus weights")

print(f"Aligned genes: {consensus_weight_matrix.shape[0]:,}")
print(f"Consensus programs: {consensus_weight_matrix.shape[1]}")
print("Frozen gene order and weights: VERIFIED")

display(consensus_weight_matrix.head())

Aligned genes: 2,389
Consensus programs: 3
Frozen gene order and weights: VERIFIED


consensus_program_id,CONSENSUS_TX_01,CONSENSUS_TX_02,CONSENSUS_TX_03
gene_symbol,,,
AADAC,-0.012754,0.000683,-0.019402
AARD,-0.002949,0.032478,-0.012021
ABCA3,-0.009322,0.027796,0.002974
ABCB1,0.001332,0.010226,-0.011124
ABCC2,-0.001405,-0.005727,-0.026280


In [7]:
# =============================================================================
# Resolve CPTAC-3 STAR Counts inputs
# =============================================================================
STAR_ROOT = Paths.root / "data" / "raw" / "cptac3" / "rnaseq_star_counts"

rna_sources = cohort_113[
    ["case_uuid", "file_id", "filename", "tcga_lineage"]
].copy()

rna_sources["star_path"] = [
    STAR_ROOT / file_id / filename
    for file_id, filename in zip(
        rna_sources["file_id"],
        rna_sources["filename"],
    )
]

missing_files = rna_sources.loc[
    ~rna_sources["star_path"].map(lambda path: path.is_file())
]

if not missing_files.empty:
    raise FileNotFoundError(
        f"Missing CPTAC-3 STAR Counts files: {len(missing_files)}"
    )

print(f"CPTAC-3 RNA-seq files available: {len(rna_sources):,}")
print(f"Cancer lineages: {rna_sources['tcga_lineage'].nunique()}")
print("STAR Counts source paths: RESOLVED")

CPTAC-3 RNA-seq files available: 727
Cancer lineages: 7
STAR Counts source paths: RESOLVED


In [8]:
# =============================================================================
# Inspect STAR Counts gene annotations
# =============================================================================

reference_star_path = rna_sources["star_path"].iloc[0]

star_gene_annotation = pd.read_csv(
    reference_star_path,
    sep="\t",
    comment="#",
    usecols=["gene_id", "gene_type"],
    dtype="string",
)

star_gene_annotation["ensembl_gene_id"] = (
    star_gene_annotation["gene_id"]
    .str.replace(r"\.\d+$", "", regex=True)
    .str.strip()
)

protein_coding_annotation = star_gene_annotation.loc[
    star_gene_annotation["gene_type"].eq("protein_coding")
].copy()

frozen_gene_ids = (
    shared_gene_universe["tumor_gene_id"]
    .str.replace(r"\.\d+$", "", regex=True)
    .str.strip()
)

print(f"Annotated STAR genes: {len(star_gene_annotation):,}")
print(f"Protein-coding genes: {len(protein_coding_annotation):,}")
print(f"Frozen reference genes: {len(frozen_gene_ids):,}")
print(
    "Frozen genes annotated as protein-coding:",
    protein_coding_annotation["ensembl_gene_id"]
    .isin(frozen_gene_ids)
    .sum(),
)

Annotated STAR genes: 60,664
Protein-coding genes: 19,962
Frozen reference genes: 2,389
Frozen genes annotated as protein-coding: 2389


In [9]:
# =============================================================================
# Build CPTAC-3 protein-coding count matrix
# =============================================================================
protein_coding_ids = protein_coding_annotation["ensembl_gene_id"].tolist()

count_values = np.empty(
    (len(protein_coding_ids), len(rna_sources)),
    dtype=np.int64,
)

for sample_index, source in enumerate(rna_sources.itertuples(index=False)):
    star_counts = pd.read_csv(
        source.star_path,
        sep="\t",
        comment="#",
        usecols=["gene_id", "unstranded"],
        dtype={"gene_id": "string", "unstranded": "int64"},
    )

    star_counts["gene_id"] = (
        star_counts["gene_id"]
        .str.replace(r"\.\d+$", "", regex=True)
        .str.strip()
    )

    aligned_counts = (
        star_counts.set_index("gene_id")["unstranded"]
        .reindex(protein_coding_ids)
    )

    if aligned_counts.isna().any():
        raise ValueError(f"Missing protein-coding genes: {source.file_id}")

    count_values[:, sample_index] = aligned_counts.to_numpy(
        dtype=np.int64
    )

protein_coding_counts = pd.DataFrame(
    count_values,
    index=pd.Index(protein_coding_ids, name="ensembl_gene_id"),
    columns=pd.Index(rna_sources["file_id"], name="file_id"),
)

print(f"Protein-coding genes: {protein_coding_counts.shape[0]:,}")
print(f"CPTAC-3 samples: {protein_coding_counts.shape[1]:,}")
print(f"Count representation: unstranded")
print(f"Matrix dtype: {protein_coding_counts.dtypes.iloc[0]}")

Protein-coding genes: 19,962
CPTAC-3 samples: 727
Count representation: unstranded
Matrix dtype: int64


In [10]:
# =============================================================================
# Export CPTAC-3 raw counts for R normalization
# =============================================================================

counts_path = Paths.expression / "800_cptac3_protein_coding_raw_counts.h5"
genes_path = Paths.expression / "800_cptac3_protein_coding_gene_metadata.csv"
samples_path = Paths.expression / "800_cptac3_sample_metadata.csv"

gene_metadata = protein_coding_annotation[
    ["gene_id", "ensembl_gene_id", "gene_type"]
].reset_index(drop=True)

sample_metadata = rna_sources[
    ["file_id", "case_uuid", "tcga_lineage"]
].reset_index(drop=True)

if not np.array_equal(
    gene_metadata["ensembl_gene_id"], protein_coding_counts.index
):
    raise ValueError("Gene order mismatch")

if not np.array_equal(
    sample_metadata["file_id"], protein_coding_counts.columns
):
    raise ValueError("Sample order mismatch")

Paths.expression.mkdir(parents=True, exist_ok=True)

gene_metadata.to_csv(genes_path, index=False)
sample_metadata.to_csv(samples_path, index=False)

with h5py.File(counts_path, "w") as h5_file:
    dataset = h5_file.create_dataset(
        "counts",
        data=protein_coding_counts.to_numpy(copy=False),
        compression="gzip",
        compression_opts=4,
        chunks=(256, 256),
        shuffle=True,
    )
    dataset.attrs["orientation"] = "genes_x_samples"

print(f"Count matrix: {protein_coding_counts.shape}")
print(f"Gene metadata: {len(gene_metadata):,} rows")
print(f"Sample metadata: {len(sample_metadata):,} rows")
print("Python–R exchange files saved.")

Count matrix: (19962, 727)
Gene metadata: 19,962 rows
Sample metadata: 727 rows
Python–R exchange files saved.


In [11]:
# =============================================================================
# Load TMM-normalized CPTAC-3 expression
# =============================================================================
CPTAC3_LOGCPM_PATH = (
    Paths.expression / "800_cptac3_frozen_genes_tmm_logcpm.h5"
)
CPTAC3_SAMPLE_METADATA_PATH = (
    Paths.expression / "800_cptac3_sample_metadata.csv"
)

r_sample_metadata = pd.read_csv(CPTAC3_SAMPLE_METADATA_PATH)

if not r_sample_metadata["file_id"].equals(
    rna_sources["file_id"].reset_index(drop=True)
):
    raise ValueError("Sample order differs from the R normalization input")

with h5py.File(CPTAC3_LOGCPM_PATH, "r") as h5_file:
    dataset = h5_file["logcpm"]
    expected_shape = (len(shared_gene_universe), len(rna_sources))

    if dataset.shape != expected_shape:
        raise ValueError("Unexpected normalized expression dimensions")

    orientation = dataset.attrs["orientation"].item().decode("utf-8")

    if orientation != "genes_x_samples":
        raise ValueError("Unexpected normalized expression orientation")

    logcpm_values = dataset[:]

cptac3_logcpm = pd.DataFrame(
    logcpm_values,
    index=pd.Index(shared_gene_universe["gene_symbol"], name="gene_symbol"),
    columns=pd.Index(rna_sources["file_id"], name="file_id"),
)

print(f"Normalized expression: {cptac3_logcpm.shape}")
print(f"Matrix dtype: {cptac3_logcpm.dtypes.iloc[0]}")
print(f"Finite values: {np.isfinite(logcpm_values).all()}")

Normalized expression: (2389, 727)
Matrix dtype: float64
Finite values: True


In [12]:
# =============================================================================
# Assess gene expression variability
# =============================================================================
gene_sd = cptac3_logcpm.std(axis=1, ddof=1)

gene_range = np.ptp(cptac3_logcpm.to_numpy(), axis=1)

constant_genes = cptac3_logcpm.index[gene_range == 0]
zero_sd_genes = gene_sd.index[gene_sd == 0]

print(f"Frozen genes: {len(gene_sd):,}")
print(f"Genes with constant expression: {len(constant_genes)}")
print(f"Genes with zero standard deviation: {len(zero_sd_genes)}")
print(f"Minimum gene SD: {gene_sd.min():.8f}")
print(f"Median gene SD: {gene_sd.median():.4f}")

if len(constant_genes) > 0:
    display(pd.DataFrame({"constant_gene": constant_genes}))

Frozen genes: 2,389
Genes with constant expression: 0
Genes with zero standard deviation: 0
Minimum gene SD: 0.71618470
Median gene SD: 1.6544


In [13]:
# =============================================================================
# Standardize CPTAC-3 gene expression
# =============================================================================
expression_values = cptac3_logcpm.to_numpy(dtype=np.float64)

gene_means = expression_values.mean(axis=1, keepdims=True)
gene_stds = expression_values.std(axis=1, ddof=1, keepdims=True)

# Match the samples-by-genes orientation used for Phase 4 projection.
standardized_expression = (
    (expression_values - gene_means) / gene_stds
).T

print(f"Standardized expression: {standardized_expression.shape}")
print(f"Finite values: {np.isfinite(standardized_expression).all()}")
print(
    "Maximum absolute gene mean:",
    f"{np.abs(standardized_expression.mean(axis=0)).max():.2e}",
)
print(
    "Maximum deviation from unit gene SD:",
    f"{np.abs(standardized_expression.std(axis=0, ddof=1) - 1).max():.2e}",
)

Standardized expression: (727, 2389)


Finite values: True
Maximum absolute gene mean: 3.81e-14
Maximum deviation from unit gene SD: 7.77e-15


In [14]:
# =============================================================================
# Project frozen consensus transcriptomic programs
# =============================================================================
weight_values = consensus_weight_matrix.to_numpy(dtype=np.float64)

projected_scores = standardized_expression @ weight_values

score_means = projected_scores.mean(axis=0, keepdims=True)
score_stds = projected_scores.std(axis=0, ddof=1, keepdims=True)

if not np.isfinite(score_stds).all() or (score_stds == 0).any():
    raise ValueError("Program score standardization is undefined")

standardized_scores = (projected_scores - score_means) / score_stds

cptac3_program_scores = pd.DataFrame(
    standardized_scores,
    index=pd.Index(rna_sources["file_id"], name="file_id"),
    columns=consensus_weight_matrix.columns,
)

print(f"Projected program scores: {cptac3_program_scores.shape}")
print(f"Finite values: {np.isfinite(standardized_scores).all()}")

display(
    cptac3_program_scores.agg(["mean", "std"]).round(6)
)

Projected program scores: (727, 3)
Finite values: True


consensus_program_id,CONSENSUS_TX_01,CONSENSUS_TX_02,CONSENSUS_TX_03
mean,0.0,0.0,-0.0
std,1.0,1.0,1.0


In [15]:
# =============================================================================
# Assess program-score variability by lineage
# =============================================================================
lineage_program_scores = rna_sources[
    ["file_id", "case_uuid", "tcga_lineage"]
].merge(
    cptac3_program_scores.reset_index(),
    on="file_id",
    how="left",
    validate="one_to_one",
)

lineage_score_sd = (
    lineage_program_scores.groupby("tcga_lineage")[list(PROGRAM_IDS)]
    .std(ddof=1)
)

lineage_score_sd.insert(
    0,
    "n_samples",
    lineage_program_scores.groupby("tcga_lineage").size(),
)

print("Within-lineage program-score variability:")
display(lineage_score_sd.round(4))

if not np.isfinite(lineage_score_sd[list(PROGRAM_IDS)].to_numpy()).all():
    raise ValueError("Undefined within-lineage program-score variability")

Within-lineage program-score variability:


,n_samples,CONSENSUS_TX_01,CONSENSUS_TX_02,CONSENSUS_TX_03
tcga_lineage,,,,
TCGA-HNSC,36,0.9701,1.0085,0.8600
TCGA-LUAD,178,0.6278,0.4449,0.6566
TCGA-LUSC,89,0.8255,0.7807,0.8668
TCGA-PAAD,88,0.7471,0.4733,0.7432
TCGA-SKCM,138,1.3282,0.9343,0.8532
TCGA-STAD,142,0.9032,0.7935,1.1307
TCGA-UCEC,56,0.9232,0.7451,0.9347


In [16]:
# =============================================================================
# Load TCGA transcriptomic reference metadata
# =============================================================================
TCGA_REFERENCE_IDS = {
    "expression": "phase2.205.rna_program_discovery_expression",
    "genes": "phase2.205.rna_program_discovery_gene_metadata",
    "samples": "phase2.205.rna_program_discovery_sample_metadata",
}

tcga_reference_paths = {
    name: resolve_artifact_path(artifact_registry, artifact_id)
    for name, artifact_id in TCGA_REFERENCE_IDS.items()
}

tcga_gene_metadata = pd.read_csv(tcga_reference_paths["genes"])
tcga_sample_metadata = pd.read_csv(tcga_reference_paths["samples"])

tcga_lineages = sorted(rna_sources["tcga_lineage"].unique())

tcga_lineage_counts = (
    tcga_sample_metadata["project_id"]
    .value_counts()
    .reindex(tcga_lineages, fill_value=0)
)

tcga_gene_ids = (
    tcga_gene_metadata["gene_id"]
    .astype("string")
    .str.replace(r"\.\d+$", "", regex=True)
)

frozen_gene_ids = (
    shared_gene_universe["tumor_gene_id"]
    .astype("string")
    .str.replace(r"\.\d+$", "", regex=True)
)

print(f"TCGA reference genes: {len(tcga_gene_metadata):,}")
print(f"TCGA reference samples: {len(tcga_sample_metadata):,}")
print(f"Frozen genes available: {frozen_gene_ids.isin(tcga_gene_ids).sum():,}")

display(tcga_lineage_counts.rename("n_tcga_samples").to_frame())

TCGA reference genes: 18,123
TCGA reference samples: 9,965
Frozen genes available: 2,389


,n_tcga_samples
project_id,
TCGA-HNSC,520
TCGA-LUAD,512
TCGA-LUSC,500
TCGA-PAAD,178
TCGA-SKCM,103
TCGA-STAD,412
TCGA-UCEC,543


In [17]:
# =============================================================================
# Load lineage-matched TCGA expression
# =============================================================================
tcga_sample_indices = np.flatnonzero(
    tcga_sample_metadata["project_id"].isin(tcga_lineages).to_numpy()
)

tcga_reference_samples = (
    tcga_sample_metadata.iloc[tcga_sample_indices]
    .reset_index(drop=True)
)

with h5py.File(tcga_reference_paths["expression"], "r") as h5_file:
    expression_dataset = h5_file["logcpm"]

    if expression_dataset.shape != (
        len(tcga_gene_metadata),
        len(tcga_sample_metadata),
    ):
        raise ValueError("TCGA expression dimensions do not match metadata")

    tcga_reference_expression = expression_dataset[:, tcga_sample_indices]

print(f"TCGA genes: {tcga_reference_expression.shape[0]:,}")
print(f"Selected TCGA samples: {tcga_reference_expression.shape[1]:,}")
print(f"Selected lineages: {len(tcga_lineages)}")
print(f"Expression dtype: {tcga_reference_expression.dtype}")

TCGA genes: 18,123
Selected TCGA samples: 2,768
Selected lineages: 7
Expression dtype: float64


In [18]:
# =============================================================================
# Align TCGA expression with the frozen gene universe
# =============================================================================

# Verify that metadata rows match the HDF5 sample-column order.
expected_columns = np.arange(len(tcga_sample_metadata))

if not np.array_equal(
    tcga_sample_metadata["final_sample_column_index"].to_numpy(),
    expected_columns,
):
    raise ValueError("TCGA sample metadata are not aligned with expression")

# Match genes by stable Ensembl identifiers, not gene symbols.
tcga_gene_index = pd.Index(tcga_gene_ids)
frozen_gene_index = pd.Index(frozen_gene_ids)

if (
    tcga_gene_index.hasnans
    or frozen_gene_index.hasnans
    or not tcga_gene_index.is_unique
    or not frozen_gene_index.is_unique
):
    raise ValueError("Invalid or duplicated gene identifiers")

gene_positions = tcga_gene_index.get_indexer(frozen_gene_index)

if (gene_positions < 0).any():
    raise ValueError("Frozen genes missing from TCGA expression")

# Preserve the exact frozen gene order.
tcga_frozen_logcpm = pd.DataFrame(
    tcga_reference_expression[gene_positions, :],
    index=pd.Index(gene_order, name="gene_symbol"),
    columns=pd.Index(
        tcga_reference_samples["sample_submitter_id"],
        name="sample_submitter_id",
    ),
)

if not tcga_frozen_logcpm.index.equals(cptac3_logcpm.index):
    raise ValueError("TCGA and CPTAC-3 frozen gene orders differ")

if not tcga_frozen_logcpm.columns.is_unique:
    raise ValueError("Duplicated TCGA sample identifiers")

if tcga_reference_samples["case_submitter_id"].duplicated().any():
    raise ValueError("Multiple TCGA samples from the same case")

if not np.isfinite(tcga_frozen_logcpm.to_numpy()).all():
    raise ValueError("Nonfinite TCGA expression values")

print(f"Aligned TCGA expression: {tcga_frozen_logcpm.shape}")
print(f"Aligned CPTAC-3 expression: {cptac3_logcpm.shape}")
print(f"Frozen genes matched: {len(gene_positions):,}")

Aligned TCGA expression: (2389, 2768)
Aligned CPTAC-3 expression: (2389, 727)
Frozen genes matched: 2,389


In [19]:
# =============================================================================
# Assess lineage-specific expression variability
# =============================================================================

cohort_inputs = {
    "TCGA": (
        tcga_frozen_logcpm,
        tcga_reference_samples,
        "sample_submitter_id",
        "case_submitter_id",
        "project_id",
    ),
    "CPTAC-3": (
        cptac3_logcpm,
        rna_sources,
        "file_id",
        "case_uuid",
        "tcga_lineage",
    ),
}

preflight_rows = []

for cohort, (expression, metadata, sample_col, case_col, lineage_col) in (
    cohort_inputs.items()
):
    if not expression.columns.equals(pd.Index(metadata[sample_col])):
        raise ValueError(f"{cohort}: expression/sample order mismatch")

    for lineage in tcga_lineages:
        selected = metadata.loc[metadata[lineage_col] == lineage]
        sample_ids = selected[sample_col].tolist()
        values = expression.loc[:, sample_ids].to_numpy(dtype=np.float64)

        finite_genes = np.isfinite(values).all(axis=1)
        gene_sd = np.full(values.shape[0], np.nan)

        if len(sample_ids) > 1:
            gene_sd[finite_genes] = values[finite_genes].std(
                axis=1, ddof=1
            )

        n_zero_sd = int(np.count_nonzero(gene_sd[finite_genes] == 0))
        n_nonfinite = int(np.count_nonzero(~finite_genes))
        n_cases = selected[case_col].nunique()

        preflight_rows.append(
            {
                "lineage": lineage,
                "cohort": cohort,
                "n_samples": len(sample_ids),
                "n_cases": n_cases,
                "nonfinite_genes": n_nonfinite,
                "zero_sd_genes": n_zero_sd,
                "min_gene_sd": np.nanmin(gene_sd),
                "technically_evaluable": (
                    len(sample_ids) > 1
                    and n_cases == len(sample_ids)
                    and selected[case_col].notna().all()
                    and n_nonfinite == 0
                    and n_zero_sd == 0
                ),
            }
        )

lineage_preflight = pd.DataFrame(preflight_rows)

display(
    lineage_preflight.set_index(["lineage", "cohort"]).sort_index().round(6)
)

n_samples  n_cases  nonfinite_genes  zero_sd_genes  \
lineage   cohort                                                        
TCGA-HNSC CPTAC-3         36       36                0              0   
          TCGA           520      520                0              0   
TCGA-LUAD CPTAC-3        178      178                0              0   
          TCGA           512      512                0              0   
TCGA-LUSC CPTAC-3         89       89                0              0   
          TCGA           500      500                0              0   
TCGA-PAAD CPTAC-3         88       88                0              0   
          TCGA           178      178                0              0   
TCGA-SKCM CPTAC-3        138      138                0              0   
          TCGA           103      103                0              0   
TCGA-STAD CPTAC-3        142      142                0              0   
          TCGA           412      412                0              0   
TCGA-UCEC CPTAC-3         56       56                0              0   
          TCGA           543      543                0              0   

                   min_gene_sd  technically_evaluable  
lineage   cohort                                       
TCGA-HNSC CPTAC-3     0.494263                   True  
          TCGA        0.665547                   True  
TCGA-LUAD CPTAC-3     0.437288                   True  
          TCGA        0.416059                   True  
TCGA-LUSC CPTAC-3     0.354170                   True  
          TCGA        0.366417                   True  
TCGA-PAAD CPTAC-3     0.207936                   True  
          TCGA        0.514597                   True  
TCGA-SKCM CPTAC-3     0.330953                   True  
          TCGA        0.278483                   True  
TCGA-STAD CPTAC-3     0.436618                   True  
          TCGA        0.770464                   True  
TCGA-UCEC CPTAC-3     0.478592                   True  
          TCGA        0.338074                   True

In [20]:
# =============================================================================
# Prepare lineage-specific expression matrices
# =============================================================================

frozen_index = consensus_weight_matrix.index

for cohort, expression in {
    "TCGA": tcga_frozen_logcpm,
    "CPTAC-3": cptac3_logcpm,
}.items():
    if not expression.index.equals(frozen_index):
        raise ValueError(f"{cohort}: frozen gene order mismatch")

if not lineage_preflight["technically_evaluable"].all():
    raise ValueError("Lineage-specific variability preflight failed")

lineage_expression = {}
matrix_summary = []

for lineage in tcga_lineages:
    lineage_expression[lineage] = {}

    for cohort, expression, metadata, lineage_col, sample_col in (
        (
            "TCGA",
            tcga_frozen_logcpm,
            tcga_reference_samples,
            "project_id",
            "sample_submitter_id",
        ),
        (
            "CPTAC-3",
            cptac3_logcpm,
            rna_sources,
            "tcga_lineage",
            "file_id",
        ),
    ):
        sample_ids = metadata.loc[
            metadata[lineage_col] == lineage, sample_col
        ]

        values = expression.loc[:, sample_ids].to_numpy(
            dtype=np.float64
        ).T

        lineage_expression[lineage][cohort] = values

        matrix_summary.append(
            {
                "lineage": lineage,
                "cohort": cohort,
                "n_tumors": values.shape[0],
                "n_genes": values.shape[1],
            }
        )

lineage_matrix_summary = pd.DataFrame(matrix_summary)

display(
    lineage_matrix_summary.set_index(
        ["lineage", "cohort"]
    ).sort_index()
)

n_tumors  n_genes
lineage   cohort                    
TCGA-HNSC CPTAC-3        36     2389
          TCGA          520     2389
TCGA-LUAD CPTAC-3       178     2389
          TCGA          512     2389
TCGA-LUSC CPTAC-3        89     2389
          TCGA          500     2389
TCGA-PAAD CPTAC-3        88     2389
          TCGA          178     2389
TCGA-SKCM CPTAC-3       138     2389
          TCGA          103     2389
TCGA-STAD CPTAC-3       142     2389
          TCGA          412     2389
TCGA-UCEC CPTAC-3        56     2389
          TCGA          543     2389

In [21]:
# =============================================================================
# Prepare frozen gene-pair weights for structural analysis
# =============================================================================

n_genes = len(consensus_weight_matrix)
pair_i, pair_j = np.triu_indices(n_genes, k=1)

edge_weights = {}
edge_signs = {}
weight_summary = []

for program in PROGRAM_IDS:
    gene_weights = consensus_weight_matrix[program].to_numpy(
        dtype=np.float64
    )

    pair_products = gene_weights[pair_i] * gene_weights[pair_j]
    pair_magnitudes = np.abs(pair_products)
    total_weight = pair_magnitudes.sum()

    if not np.isfinite(total_weight) or total_weight <= 0:
        raise ValueError(f"Invalid pair weights: {program}")

    normalized_weights = pair_magnitudes / total_weight

    edge_weights[program] = normalized_weights
    edge_signs[program] = np.sign(pair_products).astype(np.int8)

    weight_summary.append(
        {
            "program": program,
            "gene_pairs": len(pair_magnitudes),
            "nonzero_weight_pairs": np.count_nonzero(pair_magnitudes),
            "effective_weighted_pairs": (
                1.0 / np.sum(normalized_weights**2)
            ),
        }
    )

pair_weight_summary = pd.DataFrame(weight_summary)

display(pair_weight_summary.round(1))

,program,gene_pairs,nonzero_weight_pairs,effective_weighted_pairs
0,CONSENSUS_TX_01,2852466,2852466,441653.6
1,CONSENSUS_TX_02,2852466,2852466,875116.9
2,CONSENSUS_TX_03,2852466,2852466,938374.8


In [22]:
# =============================================================================
# Assess within-lineage TCGA coexpression reference stability
# =============================================================================

def gene_pair_correlations(values):
    centered = values - values.mean(axis=0, keepdims=True)
    norms = np.linalg.norm(centered, axis=0)

    if not np.isfinite(norms).all() or np.any(norms <= 0):
        raise ValueError("Undefined gene correlations")

    normalized = centered / norms
    correlations = (normalized.T @ normalized)[pair_i, pair_j]

    if not np.isfinite(correlations).all():
        raise ValueError("Nonfinite gene-pair correlations")

    return correlations


def weighted_edge_pearson(reference, comparison, weights):
    ref_centered = reference - np.dot(weights, reference)
    cmp_centered = comparison - np.dot(weights, comparison)

    numerator = np.dot(weights, ref_centered * cmp_centered)
    denominator = np.sqrt(
        np.dot(weights, ref_centered**2)
        * np.dot(weights, cmp_centered**2)
    )

    if not np.isfinite(denominator) or denominator <= 0:
        raise ValueError("Undefined weighted edge concordance")

    return float(numerator / denominator)


stability_rows = []

for lineage_index, lineage in enumerate(tcga_lineages):
    tcga_values = lineage_expression[lineage]["TCGA"]
    cptac_n = len(lineage_expression[lineage]["CPTAC-3"])

    n_half = min(len(tcga_values) // 2, cptac_n)

    rng = np.random.default_rng(
        np.random.SeedSequence([20261111, lineage_index])
    )

    for split in range(8):
        indices = rng.permutation(len(tcga_values))
        left = indices[:n_half]
        right = indices[n_half : 2 * n_half]

        left_edges = gene_pair_correlations(tcga_values[left])
        right_edges = gene_pair_correlations(tcga_values[right])

        for program in PROGRAM_IDS:
            stability_rows.append(
                {
                    "lineage": lineage,
                    "program": program,
                    "split": split,
                    "n_per_half": n_half,
                    "T_internal": weighted_edge_pearson(
                        left_edges,
                        right_edges,
                        edge_weights[program],
                    ),
                }
            )

tcga_reference_stability = pd.DataFrame(stability_rows)

stability_summary = (
    tcga_reference_stability.groupby(
        ["lineage", "program", "n_per_half"]
    )["T_internal"]
    .agg(["median", "min", "max"])
    .reset_index()
)

display(stability_summary.round(4))

,lineage,program,n_per_half,median,min,max
0,TCGA-HNSC,CONSENSUS_TX_01,36,0.6863,0.6377,0.7559
1,TCGA-HNSC,CONSENSUS_TX_02,36,0.5833,0.5373,0.6100
2,TCGA-HNSC,CONSENSUS_TX_03,36,0.6032,0.5875,0.6291
3,TCGA-LUAD,CONSENSUS_TX_01,178,0.8830,0.8640,0.8919
4,TCGA-LUAD,CONSENSUS_TX_02,178,0.7859,0.7655,0.7949
5,TCGA-LUAD,CONSENSUS_TX_03,178,0.8392,0.8181,0.8441
6,TCGA-LUSC,CONSENSUS_TX_01,89,0.8382,0.8247,0.8456
7,TCGA-LUSC,CONSENSUS_TX_02,89,0.6942,0.6827,0.7155
8,TCGA-LUSC,CONSENSUS_TX_03,89,0.7428,0.7230,0.7514
9,TCGA-PAAD,CONSENSUS_TX_01,88,0.8421,0.7965,0.8607


In [23]:

# =============================================================================
# Inspect normalized CPTAC-3 expression provenance
# =============================================================================

hdf5_inventory = []

with h5py.File(CPTAC3_LOGCPM_PATH, "r") as h5_file:

    def inspect_dataset(name, obj):
        if isinstance(obj, h5py.Dataset):
            hdf5_inventory.append(
                {
                    "dataset": name,
                    "shape": obj.shape,
                    "dtype": str(obj.dtype),
                    "attributes": tuple(obj.attrs.keys()),
                }
            )

    h5_file.visititems(inspect_dataset)

    file_attributes = tuple(h5_file.attrs.keys())
    expression_attributes = tuple(
        h5_file["logcpm"].attrs.keys()
    )

display(pd.DataFrame(hdf5_inventory))

print("File attributes:", file_attributes)
print("Expression attributes:", expression_attributes)

,dataset,shape,dtype,attributes
0,logcpm,"(2389, 727)",float64,"(orientation, rhdf5-NA.OK)"


File attributes: ()
Expression attributes: ('orientation', 'rhdf5-NA.OK')


In [24]:
# =============================================================================
# Verify normalized expression alignment against raw counts
# =============================================================================

frozen_ensembl_ids = pd.Index(frozen_gene_ids.astype(str))

if not frozen_ensembl_ids.is_unique:
    raise ValueError("Duplicated frozen Ensembl identifiers")

missing_genes = frozen_ensembl_ids.difference(
    protein_coding_counts.index
)

if len(missing_genes):
    raise ValueError(
        f"Frozen genes missing from raw counts: {len(missing_genes)}"
    )

if not protein_coding_counts.columns.equals(cptac3_logcpm.columns):
    raise ValueError("Raw counts and logCPM sample orders differ")

frozen_raw_counts = protein_coding_counts.loc[
    frozen_ensembl_ids
].to_numpy()

normalized_values = cptac3_logcpm.to_numpy()

if frozen_raw_counts.shape != normalized_values.shape:
    raise ValueError("Raw and normalized matrix dimensions differ")

tolerance = 1e-10
alignment_failures = []

for sample_idx, sample_id in enumerate(cptac3_logcpm.columns):
    counts = frozen_raw_counts[:, sample_idx]
    logcpm = normalized_values[:, sample_idx]

    order = np.argsort(counts, kind="stable")

    sorted_counts = counts[order]
    sorted_logcpm = logcpm[order]

    count_differences = np.diff(sorted_counts)
    logcpm_differences = np.diff(sorted_logcpm)

    inverted = np.any(logcpm_differences < -tolerance)
    inconsistent_ties = np.any(
        (count_differences == 0)
        & (np.abs(logcpm_differences) > tolerance)
    )

    if inverted or inconsistent_ties:
        alignment_failures.append(sample_id)

alignment_summary = pd.DataFrame(
    {
        "samples_checked": [normalized_values.shape[1]],
        "samples_consistent": [
            normalized_values.shape[1] - len(alignment_failures)
        ],
        "samples_inconsistent": [len(alignment_failures)],
        "genes_compared": [normalized_values.shape[0]],
    }
)

display(alignment_summary)

if alignment_failures:
    raise ValueError(
        f"Raw-count/logCPM alignment failed for "
        f"{len(alignment_failures)} samples"
    )

,samples_checked,samples_consistent,samples_inconsistent,genes_compared
0,727,727,0,2389


In [25]:
# =============================================================================
# Define weighted structural coexpression statistics
# =============================================================================

def weighted_median(values, weights):
    order = np.argsort(values)
    sorted_values = values[order]
    cumulative_weights = np.cumsum(weights[order])

    midpoint = cumulative_weights[-1] / 2
    position = np.searchsorted(cumulative_weights, midpoint, side="left")

    return float(sorted_values[position])


def structural_statistics(tcga_edges, cptac_edges):
    if tcga_edges.shape != cptac_edges.shape:
        raise ValueError("Incompatible correlation vectors")

    if tcga_edges.size != len(pair_i):
        raise ValueError("Unexpected number of gene pairs")

    if not (
        np.isfinite(tcga_edges).all()
        and np.isfinite(cptac_edges).all()
    ):
        raise ValueError("Nonfinite gene-pair correlations")

    results = []

    for program in PROGRAM_IDS:
        weights = edge_weights[program]
        signs = edge_signs[program]

        tcga_signed = signs * tcga_edges
        cptac_signed = signs * cptac_edges

        results.append(
            {
                "program": program,
                "T": weighted_edge_pearson(
                    tcga_edges, cptac_edges, weights
                ),
                "Delta_M": np.dot(
                    weights,
                    np.abs(cptac_edges) - np.abs(tcga_edges),
                ),
                "signed_median_tcga": weighted_median(
                    tcga_signed, weights
                ),
                "signed_median_cptac": weighted_median(
                    cptac_signed, weights
                ),
                "signed_mean_tcga": float(
                    np.dot(weights, tcga_signed)
                ),
                "signed_mean_cptac": float(
                    np.dot(weights, cptac_signed)
                ),
            }
        )

    return pd.DataFrame(results)

In [26]:
# =============================================================================
# Characterize full-cohort TCGA coexpression structure
# =============================================================================

tcga_reference_edges = {}
reference_rows = []

for lineage in tcga_lineages:
    values = lineage_expression[lineage]["TCGA"]
    correlations = gene_pair_correlations(values)

    if correlations.size != len(pair_i):
        raise ValueError(f"{lineage}: unexpected number of gene pairs")

    tcga_reference_edges[lineage] = correlations

    for program in PROGRAM_IDS:
        weights = edge_weights[program]
        signed_correlations = correlations * edge_signs[program]

        reference_rows.append(
            {
                "lineage": lineage,
                "program": program,
                "n_tumors": values.shape[0],
                "signed_median": weighted_median(
                    signed_correlations, weights
                ),
                "signed_mean": float(
                    np.dot(weights, signed_correlations)
                ),
                "mean_abs_correlation": float(
                    np.dot(weights, np.abs(correlations))
                ),
            }
        )

tcga_reference_summary = pd.DataFrame(reference_rows)

display(tcga_reference_summary.round(4))

,lineage,program,n_tumors,signed_median,signed_mean,mean_abs_correlation
0,TCGA-HNSC,CONSENSUS_TX_01,520,0.0621,0.0921,0.1881
1,TCGA-HNSC,CONSENSUS_TX_02,520,0.0704,0.0780,0.1601
2,TCGA-HNSC,CONSENSUS_TX_03,520,0.0501,0.0678,0.1645
3,TCGA-LUAD,CONSENSUS_TX_01,512,0.0316,0.0663,0.1637
4,TCGA-LUAD,CONSENSUS_TX_02,512,0.0276,0.0320,0.1226
5,TCGA-LUAD,CONSENSUS_TX_03,512,0.0336,0.0492,0.1397
6,TCGA-LUSC,CONSENSUS_TX_01,500,0.0504,0.0863,0.1913
7,TCGA-LUSC,CONSENSUS_TX_02,500,0.0518,0.0596,0.1391
8,TCGA-LUSC,CONSENSUS_TX_03,500,0.0332,0.0474,0.1501
9,TCGA-PAAD,CONSENSUS_TX_01,178,0.0317,0.0627,0.2387


In [27]:
# =============================================================================
# Characterize external CPTAC-3 coexpression structure
# =============================================================================

cptac_reference_edges = {}
cptac_rows = []

for lineage in tcga_lineages:
    values = lineage_expression[lineage]["CPTAC-3"]
    correlations = gene_pair_correlations(values)

    if correlations.size != len(pair_i):
        raise ValueError(f"{lineage}: unexpected number of gene pairs")

    cptac_reference_edges[lineage] = correlations

    for program in PROGRAM_IDS:
        weights = edge_weights[program]
        signed_correlations = correlations * edge_signs[program]

        cptac_rows.append(
            {
                "lineage": lineage,
                "program": program,
                "n_tumors": values.shape[0],
                "signed_median": weighted_median(
                    signed_correlations, weights
                ),
                "signed_mean": float(
                    np.dot(weights, signed_correlations)
                ),
                "mean_abs_correlation": float(
                    np.dot(weights, np.abs(correlations))
                ),
            }
        )

cptac_structure_summary = pd.DataFrame(cptac_rows)

display(cptac_structure_summary.round(4))

,lineage,program,n_tumors,signed_median,signed_mean,mean_abs_correlation
0,TCGA-HNSC,CONSENSUS_TX_01,36,0.0637,0.0879,0.2541
1,TCGA-HNSC,CONSENSUS_TX_02,36,0.0943,0.0933,0.2357
2,TCGA-HNSC,CONSENSUS_TX_03,36,0.0447,0.0492,0.2239
3,TCGA-LUAD,CONSENSUS_TX_01,178,0.0447,0.0714,0.1670
4,TCGA-LUAD,CONSENSUS_TX_02,178,0.0193,0.0219,0.1282
5,TCGA-LUAD,CONSENSUS_TX_03,178,0.0270,0.0393,0.1425
6,TCGA-LUSC,CONSENSUS_TX_01,89,0.0591,0.0950,0.2335
7,TCGA-LUSC,CONSENSUS_TX_02,89,0.0578,0.0605,0.1711
8,TCGA-LUSC,CONSENSUS_TX_03,89,0.0506,0.0601,0.1880
9,TCGA-PAAD,CONSENSUS_TX_01,88,0.0851,0.1012,0.2074


In [28]:
# =============================================================================
# Compare lineage-matched TCGA and CPTAC-3 coexpression structure
# =============================================================================

comparison_rows = []

for lineage in tcga_lineages:
    statistics = structural_statistics(
        tcga_reference_edges[lineage],
        cptac_reference_edges[lineage],
    )

    statistics.insert(0, "lineage", lineage)
    comparison_rows.append(statistics)

external_structure = pd.concat(
    comparison_rows,
    ignore_index=True,
)

if not np.isfinite(
    external_structure.select_dtypes(include="number").to_numpy()
).all():
    raise ValueError("Nonfinite structural statistics")

if (external_structure["T"].abs() > 1 + 1e-10).any():
    raise ValueError("Weighted Pearson concordance outside [-1, 1]")

display(
    external_structure[
        [
            "lineage",
            "program",
            "T",
            "Delta_M",
            "signed_median_tcga",
            "signed_median_cptac",
            "signed_mean_tcga",
            "signed_mean_cptac",
        ]
    ].round(4)
)

,lineage,program,T,Delta_M,signed_median_tcga,signed_median_cptac,signed_mean_tcga,signed_mean_cptac
0,TCGA-HNSC,CONSENSUS_TX_01,0.7401,0.0660,0.0621,0.0637,0.0921,0.0879
1,TCGA-HNSC,CONSENSUS_TX_02,0.7084,0.0756,0.0704,0.0943,0.0780,0.0933
2,TCGA-HNSC,CONSENSUS_TX_03,0.6515,0.0593,0.0501,0.0447,0.0678,0.0492
3,TCGA-LUAD,CONSENSUS_TX_01,0.8673,0.0033,0.0316,0.0447,0.0663,0.0714
4,TCGA-LUAD,CONSENSUS_TX_02,0.7767,0.0056,0.0276,0.0193,0.0320,0.0219
5,TCGA-LUAD,CONSENSUS_TX_03,0.8226,0.0029,0.0336,0.0270,0.0492,0.0393
6,TCGA-LUSC,CONSENSUS_TX_01,0.8893,0.0422,0.0504,0.0591,0.0863,0.0950
7,TCGA-LUSC,CONSENSUS_TX_02,0.8076,0.0320,0.0518,0.0578,0.0596,0.0605
8,TCGA-LUSC,CONSENSUS_TX_03,0.8315,0.0379,0.0332,0.0506,0.0474,0.0601
9,TCGA-PAAD,CONSENSUS_TX_01,0.6898,-0.0314,0.0317,0.0851,0.0627,0.1012


In [29]:

# =============================================================================
# Define joint tumor-level bootstrap for structural statistics
# =============================================================================

BOOTSTRAP_SEED = 20261112
BOOTSTRAP_VALID_DRAWS = 1000
BOOTSTRAP_MAX_ATTEMPTS = 1100


def bootstrap_lineage(lineage):
    """Estimate structural uncertainty from independent cohort resampling."""

    lineage_index = list(tcga_lineages).index(lineage)

    rng = np.random.default_rng(
        np.random.SeedSequence([BOOTSTRAP_SEED, lineage_index, 0])
    )

    tcga_values = lineage_expression[lineage]["TCGA"]
    cptac_values = lineage_expression[lineage]["CPTAC-3"]

    n_tcga = tcga_values.shape[0]
    n_cptac = cptac_values.shape[0]

    accepted = []
    failures = []
    n_valid = 0

    for attempt in range(1, BOOTSTRAP_MAX_ATTEMPTS + 1):
        if n_valid == BOOTSTRAP_VALID_DRAWS:
            break

        tcga_indices = rng.integers(n_tcga, size=n_tcga)
        cptac_indices = rng.integers(n_cptac, size=n_cptac)

        try:
            tcga_edges = gene_pair_correlations(
                tcga_values[tcga_indices]
            )
            cptac_edges = gene_pair_correlations(
                cptac_values[cptac_indices]
            )

            estimates = structural_statistics(
                tcga_edges,
                cptac_edges,
            )

            numeric_values = estimates.select_dtypes(
                include="number"
            ).to_numpy()

            if not np.isfinite(numeric_values).all():
                raise ValueError("Nonfinite structural statistic")

        except (ValueError, FloatingPointError) as error:
            failures.append(
                {
                    "lineage": lineage,
                    "attempt": attempt,
                    "reason": str(error),
                }
            )
            continue

        n_valid += 1

        estimates.insert(0, "replicate", n_valid)
        estimates.insert(0, "lineage", lineage)

        accepted.append(estimates)

    if n_valid != BOOTSTRAP_VALID_DRAWS:
        raise RuntimeError(
            f"{lineage}: only {n_valid} valid bootstrap draws "
            f"after {BOOTSTRAP_MAX_ATTEMPTS} attempts"
        )

    bootstrap_estimates = pd.concat(
        accepted,
        ignore_index=True,
    )

    failure_log = pd.DataFrame(
        failures,
        columns=["lineage", "attempt", "reason"],
    )

    return bootstrap_estimates, failure_log

In [30]:
# =============================================================================
# Run and checkpoint lineage-specific tumor bootstraps
# =============================================================================

import os
from time import perf_counter

bootstrap_dir = Paths.interim / "phase8" / "bootstrap"
bootstrap_dir.mkdir(parents=True, exist_ok=True)


def validate_bootstrap_output(results, lineage):
    expected_rows = BOOTSTRAP_VALID_DRAWS * len(PROGRAM_IDS)

    required = {
        "lineage",
        "program",
        "replicate",
        "T",
        "Delta_M",
        "signed_median_tcga",
        "signed_median_cptac",
        "signed_mean_tcga",
        "signed_mean_cptac",
    }

    if not required.issubset(results.columns):
        raise ValueError(f"{lineage}: missing bootstrap columns")

    if len(results) != expected_rows:
        raise ValueError(f"{lineage}: incomplete bootstrap output")

    if not results["lineage"].eq(lineage).all():
        raise ValueError(f"{lineage}: inconsistent lineage identifiers")

    if set(results["program"]) != set(PROGRAM_IDS):
        raise ValueError(f"{lineage}: unexpected program identifiers")

    if results.duplicated(["program", "replicate"]).any():
        raise ValueError(f"{lineage}: duplicated bootstrap draws")

    for program in PROGRAM_IDS:
        replicates = results.loc[
            results["program"] == program, "replicate"
        ]

        if set(replicates) != set(range(1, BOOTSTRAP_VALID_DRAWS + 1)):
            raise ValueError(f"{lineage}: incomplete replicates")

    metric_columns = sorted(required - {"lineage", "program", "replicate"})
    numeric = results[metric_columns].to_numpy(dtype=np.float64)

    if not np.isfinite(numeric).all():
        raise ValueError(f"{lineage}: nonfinite bootstrap results")


execution_rows = []

for lineage in tcga_lineages:
    result_file = bootstrap_dir / f"{lineage}_bootstrap.csv"
    failure_file = bootstrap_dir / f"{lineage}_failures.csv"

    if result_file.exists():
        existing = pd.read_csv(result_file)
        validate_bootstrap_output(existing, lineage)

        execution_rows.append(
            {
                "lineage": lineage,
                "status": "existing",
                "valid_replicates": BOOTSTRAP_VALID_DRAWS,
                "failed_attempts": np.nan,
                "elapsed_minutes": np.nan,
            }
        )

        print(f"{lineage}: existing output validated", flush=True)
        continue

    print(f"{lineage}: starting bootstrap", flush=True)
    start = perf_counter()

    results, failures = bootstrap_lineage(lineage)
    validate_bootstrap_output(results, lineage)

    elapsed = (perf_counter() - start) / 60

    failure_tmp = failure_file.with_suffix(".csv.tmp")
    result_tmp = result_file.with_suffix(".csv.tmp")

    failures.to_csv(failure_tmp, index=False)
    os.replace(failure_tmp, failure_file)

    results.to_csv(result_tmp, index=False)
    os.replace(result_tmp, result_file)

    execution_rows.append(
        {
            "lineage": lineage,
            "status": "completed",
            "valid_replicates": BOOTSTRAP_VALID_DRAWS,
            "failed_attempts": len(failures),
            "elapsed_minutes": elapsed,
        }
    )

    print(
        f"{lineage}: completed | "
        f"{BOOTSTRAP_VALID_DRAWS} valid draws | "
        f"{len(failures)} failures | "
        f"{elapsed:.2f} min",
        flush=True,
    )

bootstrap_execution_summary = pd.DataFrame(execution_rows)

display(bootstrap_execution_summary.round(2))


TCGA-HNSC: existing output validated
TCGA-LUAD: existing output validated
TCGA-LUSC: existing output validated
TCGA-PAAD: existing output validated
TCGA-SKCM: existing output validated
TCGA-STAD: existing output validated
TCGA-UCEC: existing output validated


,lineage,status,valid_replicates,failed_attempts,elapsed_minutes
0,TCGA-HNSC,existing,1000,NaN,NaN
1,TCGA-LUAD,existing,1000,NaN,NaN
2,TCGA-LUSC,existing,1000,NaN,NaN
3,TCGA-PAAD,existing,1000,NaN,NaN
4,TCGA-SKCM,existing,1000,NaN,NaN
5,TCGA-STAD,existing,1000,NaN,NaN
6,TCGA-UCEC,existing,1000,NaN,NaN


In [31]:
# =============================================================================
# Summarize tumor bootstrap distributions by lineage and program
# =============================================================================

bootstrap_metrics = [
    "T",
    "Delta_M",
    "signed_median_tcga",
    "signed_median_cptac",
    "signed_mean_tcga",
    "signed_mean_cptac",
]
bootstrap_keys = ["lineage", "program", "statistic"]

bootstrap_draws = pd.concat(
    [
        pd.read_csv(bootstrap_dir / f"{lineage}_bootstrap.csv")
        for lineage in bootstrap_execution_summary["lineage"]
    ],
    ignore_index=True,
)

bootstrap_percentiles = (
    bootstrap_draws.melt(
        id_vars=["lineage", "program"],
        value_vars=bootstrap_metrics,
        var_name="statistic",
        value_name="bootstrap_value",
    )
    .groupby(bootstrap_keys)["bootstrap_value"]
    .quantile([0.025, 0.50, 0.975])
    .unstack()
    .rename(columns={0.025: "p025", 0.50: "bootstrap_median", 0.975: "p975"})
    .reset_index()
)

bootstrap_summary = (
    external_structure.melt(
        id_vars=["lineage", "program"],
        value_vars=bootstrap_metrics,
        var_name="statistic",
        value_name="estimate",
    )
    .merge(
        bootstrap_percentiles,
        on=bootstrap_keys,
        how="left",
        validate="one_to_one",
    )
    .sort_values(["lineage", "program"], kind="stable")
    .reset_index(drop=True)
)

if bootstrap_summary[["p025", "bootstrap_median", "p975"]].isna().any().any():
    raise ValueError("Missing bootstrap percentiles for one or more estimates.")

bootstrap_summary["interval_width"] = (
    bootstrap_summary["p975"] - bootstrap_summary["p025"]
)
bootstrap_summary["median_shift"] = (
    bootstrap_summary["bootstrap_median"] - bootstrap_summary["estimate"]
)
bootstrap_summary["estimate_outside_percentiles"] = ~bootstrap_summary[
    "estimate"
].between(bootstrap_summary["p025"], bootstrap_summary["p975"])

display(
    bootstrap_summary.loc[
        bootstrap_summary["statistic"].isin(["T", "Delta_M"])
    ].round(4)
)

,lineage,program,statistic,estimate,p025,bootstrap_median,p975,interval_width,median_shift,estimate_outside_percentiles
0,TCGA-HNSC,CONSENSUS_TX_01,T,0.7401,0.5775,0.6541,0.7112,0.1338,-0.0860,True
1,TCGA-HNSC,CONSENSUS_TX_01,Delta_M,0.0660,0.0729,0.0974,0.1323,0.0594,0.0314,True
6,TCGA-HNSC,CONSENSUS_TX_02,T,0.7084,0.5237,0.6057,0.6622,0.1385,-0.1027,True
7,TCGA-HNSC,CONSENSUS_TX_02,Delta_M,0.0756,0.0788,0.1052,0.1434,0.0646,0.0296,True
12,TCGA-HNSC,CONSENSUS_TX_03,T,0.6515,0.4660,0.5514,0.6169,0.1510,-0.1002,True
13,TCGA-HNSC,CONSENSUS_TX_03,Delta_M,0.0593,0.0696,0.0922,0.1288,0.0592,0.0329,True
18,TCGA-LUAD,CONSENSUS_TX_01,T,0.8673,0.7714,0.8027,0.8265,0.0550,-0.0646,True
19,TCGA-LUAD,CONSENSUS_TX_01,Delta_M,0.0033,-0.0013,0.0108,0.0239,0.0252,0.0076,False
24,TCGA-LUAD,CONSENSUS_TX_02,T,0.7767,0.6367,0.6707,0.6992,0.0624,-0.1059,True
25,TCGA-LUAD,CONSENSUS_TX_02,Delta_M,0.0056,0.0070,0.0143,0.0218,0.0148,0.0087,True


In [32]:
# =============================================================================
# Assess Delta_M sensitivity to equal tumor sample sizes
# =============================================================================

MATCHED_N_DRAWS = 32
MATCHED_N_SEED = 20261112

# Use a separate random stream for the matched-size sensitivity.
matched_n_rng = np.random.default_rng(
    np.random.SeedSequence(MATCHED_N_SEED, spawn_key=(800, 1))
)
matched_n_rows = []

for lineage in sorted(external_structure["lineage"].unique()):
    tcga_values = lineage_expression[lineage]["TCGA"]
    cptac_values = lineage_expression[lineage]["CPTAC-3"]
    n_matched = min(len(tcga_values), len(cptac_values))

    if len(tcga_values) > len(cptac_values):
        subsampled_cohort = "TCGA"
        larger_values = tcga_values
        fixed_abs_edges = np.abs(cptac_reference_edges[lineage])
    else:
        subsampled_cohort = "CPTAC-3"
        larger_values = cptac_values
        fixed_abs_edges = np.abs(tcga_reference_edges[lineage])

    for draw in range(1, MATCHED_N_DRAWS + 1):
        indices = matched_n_rng.choice(
            len(larger_values), size=n_matched, replace=False
        )
        sampled_abs_edges = np.abs(gene_pair_correlations(larger_values[indices]))

        if subsampled_cohort == "TCGA":
            edge_delta = fixed_abs_edges - sampled_abs_edges
        else:
            edge_delta = sampled_abs_edges - fixed_abs_edges

        if not np.isfinite(edge_delta).all():
            raise ValueError(f"{lineage}, matched-n draw {draw}: nonfinite edges.")

        for program in PROGRAM_IDS:
            matched_n_rows.append(
                {
                    "lineage": lineage,
                    "program": program,
                    "draw": draw,
                    "n_matched": n_matched,
                    "subsampled_cohort": subsampled_cohort,
                    "Delta_M": np.dot(edge_weights[program], edge_delta),
                }
            )

matched_n_draws = pd.DataFrame(matched_n_rows)
matched_n_summary = (
    matched_n_draws.groupby(
        ["lineage", "program", "n_matched", "subsampled_cohort"],
        as_index=False,
    )
    .agg(
        matched_median=("Delta_M", "median"),
        matched_min=("Delta_M", "min"),
        matched_max=("Delta_M", "max"),
    )
    .merge(
        external_structure[["lineage", "program", "Delta_M"]].rename(
            columns={"Delta_M": "full_cohort_Delta_M"}
        ),
        on=["lineage", "program"],
        validate="one_to_one",
    )
)

display(matched_n_summary.round(4))

,lineage,program,n_matched,subsampled_cohort,matched_median,matched_min,matched_max,full_cohort_Delta_M
0,TCGA-HNSC,CONSENSUS_TX_01,36,TCGA,0.0207,-0.0026,0.0403,0.0660
1,TCGA-HNSC,CONSENSUS_TX_02,36,TCGA,0.0269,-0.0038,0.0469,0.0756
2,TCGA-HNSC,CONSENSUS_TX_03,36,TCGA,0.0069,-0.0150,0.0314,0.0593
3,TCGA-LUAD,CONSENSUS_TX_01,178,TCGA,-0.0056,-0.0176,-0.0015,0.0033
4,TCGA-LUAD,CONSENSUS_TX_02,178,TCGA,-0.0046,-0.0094,-0.0003,0.0056
5,TCGA-LUAD,CONSENSUS_TX_03,178,TCGA,-0.0078,-0.0131,-0.0019,0.0029
6,TCGA-LUSC,CONSENSUS_TX_01,89,TCGA,0.0208,-0.0036,0.0418,0.0422
7,TCGA-LUSC,CONSENSUS_TX_02,89,TCGA,0.0073,-0.0035,0.0168,0.0320
8,TCGA-LUSC,CONSENSUS_TX_03,89,TCGA,0.0128,0.0023,0.0244,0.0379
9,TCGA-PAAD,CONSENSUS_TX_01,88,TCGA,-0.0407,-0.0734,-0.0176,-0.0314


In [33]:
# =============================================================================
# Consolidate lineage-specific structural assessment results
# =============================================================================

result_keys = ["lineage", "program"]

structural_results = bootstrap_summary.pivot(
    index=result_keys,
    columns="statistic",
    values=["estimate", "p025", "bootstrap_median", "p975", "median_shift"],
)
structural_results.columns = [
    f"{statistic}_{summary}" for summary, statistic in structural_results.columns
]

structural_results = structural_results.reset_index().merge(
    matched_n_summary.drop(columns="full_cohort_Delta_M"),
    on=result_keys,
    how="left",
    validate="one_to_one",
)

if structural_results["matched_median"].isna().any():
    raise ValueError("Missing matched-size sensitivity for a lineage and program.")

display(
    structural_results[
        [
            "lineage",
            "program",
            "T_estimate",
            "signed_median_tcga_estimate",
            "signed_median_cptac_estimate",
            "Delta_M_estimate",
            "n_matched",
            "matched_median",
            "matched_min",
            "matched_max",
        ]
    ].round(4)
)

,lineage,program,T_estimate,signed_median_tcga_estimate,signed_median_cptac_estimate,Delta_M_estimate,n_matched,matched_median,matched_min,matched_max
0,TCGA-HNSC,CONSENSUS_TX_01,0.7401,0.0621,0.0637,0.0660,36,0.0207,-0.0026,0.0403
1,TCGA-HNSC,CONSENSUS_TX_02,0.7084,0.0704,0.0943,0.0756,36,0.0269,-0.0038,0.0469
2,TCGA-HNSC,CONSENSUS_TX_03,0.6515,0.0501,0.0447,0.0593,36,0.0069,-0.0150,0.0314
3,TCGA-LUAD,CONSENSUS_TX_01,0.8673,0.0316,0.0447,0.0033,178,-0.0056,-0.0176,-0.0015
4,TCGA-LUAD,CONSENSUS_TX_02,0.7767,0.0276,0.0193,0.0056,178,-0.0046,-0.0094,-0.0003
5,TCGA-LUAD,CONSENSUS_TX_03,0.8226,0.0336,0.0270,0.0029,178,-0.0078,-0.0131,-0.0019
6,TCGA-LUSC,CONSENSUS_TX_01,0.8893,0.0504,0.0591,0.0422,89,0.0208,-0.0036,0.0418
7,TCGA-LUSC,CONSENSUS_TX_02,0.8076,0.0518,0.0578,0.0320,89,0.0073,-0.0035,0.0168
8,TCGA-LUSC,CONSENSUS_TX_03,0.8315,0.0332,0.0506,0.0379,89,0.0128,0.0023,0.0244
9,TCGA-PAAD,CONSENSUS_TX_01,0.6898,0.0317,0.0851,-0.0314,88,-0.0407,-0.0734,-0.0176


In [34]:
# =============================================================================
# Summarize additional RNA-seq ancestry ambiguity by lineage
# =============================================================================

cptac3_ancestry = rna_sources[["case_uuid", "file_id", "tcga_lineage"]].merge(
    cohort_113[["case_uuid", "file_id", "other_ambiguous_rna_flag"]],
    on=["case_uuid", "file_id"],
    how="left",
    validate="one_to_one",
)
cptac3_ancestry["other_ambiguous_rna_flag"] = cptac3_ancestry[
    "other_ambiguous_rna_flag"
].astype("boolean")

ancestry_by_lineage = cptac3_ancestry.groupby("tcga_lineage").agg(
    n_tumors=("case_uuid", "size"),
    n_flag_available=("other_ambiguous_rna_flag", "count"),
    n_flagged=("other_ambiguous_rna_flag", "sum"),
)
ancestry_by_lineage["n_flag_missing"] = (
    ancestry_by_lineage["n_tumors"] - ancestry_by_lineage["n_flag_available"]
)
ancestry_by_lineage["pct_flagged_among_known"] = (
    100
    * ancestry_by_lineage["n_flagged"]
    / ancestry_by_lineage["n_flag_available"].replace(0, np.nan)
)

display(ancestry_by_lineage.round(1))

,n_tumors,n_flag_available,n_flagged,n_flag_missing,pct_flagged_among_known
tcga_lineage,,,,,
TCGA-HNSC,36,36,9,0,25.0
TCGA-LUAD,178,178,9,0,5.1
TCGA-LUSC,89,89,2,0,2.2
TCGA-PAAD,88,88,0,0,0.0
TCGA-SKCM,138,138,0,0,0.0
TCGA-STAD,142,142,0,0,0.0
TCGA-UCEC,56,56,15,0,26.8


In [35]:
# =============================================================================
# Assess sensitivity to excluding cases with additional ambiguous RNA-seq files
# =============================================================================

ancestry_flag = "other_ambiguous_rna_flag"
if cptac3_ancestry[ancestry_flag].isna().any():
    raise ValueError("Missing RNA-seq ancestry flags in the analytic cohort.")

ancestry_metrics = ["T", "Delta_M", "signed_median_cptac", "signed_mean_cptac"]
ancestry_results = []

for lineage, samples in cptac3_ancestry.groupby("tcga_lineage"):
    flagged = samples[ancestry_flag]
    if not flagged.any():
        continue

    retained_file_ids = samples.loc[~flagged, "file_id"]
    retained_expression = cptac3_logcpm.loc[:, retained_file_ids].to_numpy().T
    retained_edges = gene_pair_correlations(retained_expression)

    retained_statistics = structural_statistics(
        tcga_reference_edges[lineage], retained_edges
    )
    ancestry_results.append(
        retained_statistics[["program", *ancestry_metrics]]
        .rename(columns={metric: f"{metric}_retained" for metric in ancestry_metrics})
        .assign(
            lineage=lineage,
            n_cptac_full=len(samples),
            n_cptac_retained=len(retained_file_ids),
        )
    )

full_columns = [f"{metric}_estimate" for metric in ancestry_metrics]
ancestry_sensitivity = pd.concat(ancestry_results, ignore_index=True).merge(
    structural_results[["lineage", "program", *full_columns]],
    on=["lineage", "program"],
    validate="one_to_one",
)

for metric in ancestry_metrics:
    ancestry_sensitivity[f"{metric}_shift"] = (
        ancestry_sensitivity[f"{metric}_retained"]
        - ancestry_sensitivity[f"{metric}_estimate"]
    )

display(
    ancestry_sensitivity[
        [
            "lineage",
            "program",
            "n_cptac_full",
            "n_cptac_retained",
            "T_shift",
            "Delta_M_shift",
            "signed_median_cptac_shift",
            "signed_mean_cptac_shift",
        ]
    ].round(4)
)

,lineage,program,n_cptac_full,n_cptac_retained,T_shift,Delta_M_shift,signed_median_cptac_shift,signed_mean_cptac_shift
0,TCGA-HNSC,CONSENSUS_TX_01,36,27,0.0086,0.0099,0.0158,0.0147
1,TCGA-HNSC,CONSENSUS_TX_02,36,27,0.0008,0.0108,-0.0054,-0.0054
2,TCGA-HNSC,CONSENSUS_TX_03,36,27,0.0058,0.0139,0.0030,0.0015
3,TCGA-LUAD,CONSENSUS_TX_01,178,169,-0.0067,0.0017,0.0000,0.0001
4,TCGA-LUAD,CONSENSUS_TX_02,178,169,-0.0067,0.0005,-0.0005,-0.0006
5,TCGA-LUAD,CONSENSUS_TX_03,178,169,-0.0047,0.0007,-0.0001,-0.0005
6,TCGA-LUSC,CONSENSUS_TX_01,89,87,-0.0029,-0.0007,-0.0003,0.0000
7,TCGA-LUSC,CONSENSUS_TX_02,89,87,-0.0044,-0.0000,-0.0003,0.0001
8,TCGA-LUSC,CONSENSUS_TX_03,89,87,-0.0046,-0.0004,0.0002,0.0004
9,TCGA-UCEC,CONSENSUS_TX_01,56,41,0.1235,0.0284,0.0129,0.0282


In [36]:
# =============================================================================
# Summarize TCGA covariate coverage by analytic lineage
# =============================================================================

tcga_covariate_keys = ["project_id", "case_submitter_id", "sample_submitter_id"]
tcga_covariate_fields = [
    "absolute_purity",
    "leukocyte_fraction",
    "external_panimmune_proliferation_score",
    "tissue_source_site",
    "rna_plate",
    "rna_center",
]

tcga_covariates = pd.read_csv(
    Paths.metadata / "tcga_primary_tumor_multiomic_confounder_covariates.csv",
    usecols=[*tcga_covariate_keys, *tcga_covariate_fields],
)

tcga_analytic_covariates = tcga_reference_samples[tcga_covariate_keys].merge(
    tcga_covariates,
    on=tcga_covariate_keys,
    how="left",
    validate="one_to_one",
)

covariates_by_lineage = tcga_analytic_covariates.groupby("project_id")
tcga_covariate_coverage = (
    covariates_by_lineage[tcga_covariate_fields]
    .count()
    .add_prefix("n_")
)
tcga_covariate_coverage.insert(0, "n_tumors", covariates_by_lineage.size())
tcga_covariate_coverage.index.name = "lineage"

display(tcga_covariate_coverage)

,n_tumors,n_absolute_purity,n_leukocyte_fraction,n_external_panimmune_proliferation_score,n_tissue_source_site,n_rna_plate,n_rna_center
lineage,,,,,,,
TCGA-HNSC,520,507,520,514,520,520,520
TCGA-LUAD,512,499,512,454,512,512,512
TCGA-LUSC,500,492,500,485,500,500,500
TCGA-PAAD,178,158,178,151,178,178,178
TCGA-SKCM,103,103,103,103,103,103,103
TCGA-STAD,412,400,412,387,412,412,412
TCGA-UCEC,543,527,543,526,543,543,543


In [37]:
# =============================================================================
# Summarize CPTAC-3 RNA-seq source provenance by lineage
# =============================================================================

cptac3_rna_provenance = (
    cohort_113.groupby(
        ["tcga_lineage", "workflow_type", "platform"],
        dropna=False,
    )
    .size()
    .rename("n_tumors")
    .reset_index()
)

display(cptac3_rna_provenance)

,tcga_lineage,workflow_type,platform,n_tumors
0,TCGA-HNSC,STAR - Counts,Illumina,36
1,TCGA-LUAD,STAR - Counts,Illumina,178
2,TCGA-LUSC,STAR - Counts,Illumina,89
3,TCGA-PAAD,STAR - Counts,Illumina,88
4,TCGA-SKCM,STAR - Counts,Illumina,137
5,TCGA-SKCM,STAR - Counts,NaN,1
6,TCGA-STAD,STAR - Counts,Illumina,142
7,TCGA-UCEC,STAR - Counts,Illumina,56


## External structural assessment

Weighted edge-pattern concordance was positive across all 21 program–lineage
comparisons (`T = 0.561–0.895`). STAD had the highest point estimates for all
three programs. PAAD and UCEC had lower concordance across programs. Weighted
signed-coherence medians were positive in both cohorts for all 21 comparisons.

The tumor bootstrap produced 1,000 valid draws per lineage. Bootstrap medians
for `T` were below the corresponding full-cohort estimates in all 21
comparisons, and 17 point estimates were above the bootstrap 97.5th percentile.
The percentile bands describe the resampling distributions and should be read
alongside the full-cohort estimates and the observed displacement.

Equalizing tumor counts reduced the positive `Delta_M` estimates in HNSC and
UCEC across all three programs. In PAAD, `Delta_M` was negative in all 32
matched-size subsamples for each program, indicating lower estimated average
absolute coexpression in CPTAC-3 under this sensitivity analysis. Excluding
CPTAC-3 cases with additional RNA-seq files of ambiguous ancestry increased
`T` by 0.081–0.124 across the three UCEC programs. That comparison reflects
both a change in cohort composition and a reduction from 56 to 41 tumors.

Both cohorts used GDC STAR Counts. CPTAC-3 reported Illumina for 726 of 727
tumors, with platform missing for one SKCM tumor. Library preparation,
read-level quality, and batch differences remain unresolved. The CPTAC-3
inputs lacked measurements comparable to the TCGA tumor purity, leukocyte
fraction, and proliferation covariates. The results provide a descriptive
assessment of coexpression structure in an external cohort, with
interpretation specific to each lineage.